In [0]:
%sql
CREATE OR REPLACE VIEW sts_prd.`01_ipd_thering`.g_bowler_ww_sfdc_latestsnap AS
WITH bounds AS (
    -- Start of current fiscal year (Oct 1)
    SELECT add_months(date_trunc('year', add_months(current_date(), -9)), 9) AS start_curr_fy
)
SELECT
    -- snapshot
    c.SnapshotDate,

    -- opportunity grain
    c.OpportunityId,
    c.OpportunityLineItemId,
    c.OpportunityName,
    c.AccountName,
    c.OpportunityOwnerName,
    c.OpportunityType,
    -- c.BDFunnelStageDerived,

    -- product grain (FULL PRODUCT DETAIL PRESERVED)
    c.ProductId,
    c.ProductName,
    c.ProductDerived,
    c.ProductSubsetDerived,
    c.ProductLineDerived,
    c.ProductLineCodeDerived,
    c.ProductCategoryDerived,
    c.ProductCategoryCodeDerived,
    c.ProductSetDerived,
    c.ProductSetCodeDerived,
    c.ProductFamilyDerived,
    c.ProductClassification,
    c.PlatformDerived,
    c.PlatformCodeDerived,

    -- org/geo
    c.BusinessUnit,
    c.BusinessUnitDerived,
    c.SegmentDerived,
    c.RegionDerived,
    c.SubRegionDerived,
    c.CountryDerived,

    -- opp status
    c.OpportunityStage,
    c.OpportunityStageDerived,
    c.OpportunityProbability,
    c.OpportunityCloseDateDerived,
    c.OpportunityCreatedDateDerived,
    c.ForecastCategory,

    -- metrics (aggregated)
    SUM(c.AnnualAmountUSDDerived) AS AnnualAmountUSD,
    SUM(c.FiscalYearAmountUSDDerived) AS FiscalYearAmountUSD,
    SUM(c.NextYearCarryOverAmountUSDDerived) AS NextYearCarryOverAmountUSD

FROM hub_prd.s_global_marketing.sales_opportunity_monthly_snapshot_interim c
INNER JOIN (
    SELECT DISTINCT Product, Level1Desc, Level3
    FROM hub_prd.g_bpc_finance.v_reltioproduct_hier
    WHERE Level1Desc IN (
        'Alaris System','Hazardous Drug Solutions','IV Solutions',
        'Gravity and Syringe Delivery','IV Access','CME',
        'Alaris Plus','IPD OEM','BD Nexus'
    )
    AND Product IS NOT NULL
) m
    ON c.ProductLineCodeDerived = m.Level3
CROSS JOIN bounds
WHERE c.SnapshotDate >= bounds.start_curr_fy
  AND c.SnapshotDate < add_months(bounds.start_curr_fy, 12)
-- + for current month take all date added - for past month take only latest date added

GROUP BY
    c.SnapshotDate,
    c.OpportunityId,
    c.OpportunityLineItemId,
    c.OpportunityName,
    c.AccountName,
    c.OpportunityOwnerName,
    c.OpportunityType,
    -- c.BDFunnelStageDerived,
    c.ProductId,
    c.ProductName,
    c.ProductDerived,
    c.ProductSubsetDerived,
    c.ProductLineDerived,
    c.ProductLineCodeDerived,
    c.ProductCategoryDerived,
    c.ProductCategoryCodeDerived,
    c.ProductSetDerived,
    c.ProductSetCodeDerived,
    c.ProductFamilyDerived,
    c.ProductClassification,
    c.PlatformDerived,
    c.PlatformCodeDerived,
    c.BusinessUnit,
    c.BusinessUnitDerived,
    c.SegmentDerived,
    c.RegionDerived,
    c.SubRegionDerived,
    c.CountryDerived,
    c.OpportunityStage,
    c.OpportunityStageDerived,
    c.OpportunityProbability,
    c.OpportunityCloseDateDerived,
    c.OpportunityCreatedDateDerived,
    c.ForecastCategory;
